# MTM: Methods-Time Measurement
- MTM card라고 있는데 이게 가중치에 뭐에 따질게 많아서 결국 claude한테 부탁했습니다...
- 동작: 바닥에 있는 5kg짜리 상자를 들어서 10m 이동 후 높이가 60cm인 테이블 위에 놓기

In [1]:
# MTM-1 Data Card 기반 TMU 테이블
reach_table = {
    30: {'A': 9.5, 'B': 12.8, 'C': 14.1, 'D': 11.7, 'E': 8.0},
    # 필요한 거리만 우선
}

move_table = {
    30: {'A': 12.7, 'B': 13.3, 'C': 15.1, 'Bm': 9.8},
    60: {'A': 22.1, 'B': 20.4, 'C': 25.2, 'Bm': 18.2},
}

grasp_table = {'G1A': 2.0, 'G1B': 3.5, 'G4A': 7.3}
release_table = {'RL1': 2.0, 'RL2': 0.0}

# 중량보정 (5kg)
weight_correction = 1.12  # Data Card 중량보정 계수

# Walk: 15.0 TMU/pace, 1 pace ≈ 75cm → 10m = 약 13.3 paces
walk_tmu_per_pace = 17.0  # 부하 있으므로 W-PO 사용
walk_paces = 10 / 0.75

# 작업 시퀀스
sequence = [
    ('Bend',    29.0,  '허리 숙임 (바닥 상자)'),
    ('R30B',    12.8,  '상자로 손 뻗음 (30cm 가정, Case B)'),
    ('G4A',      7.3,  '상자 잡음 (큰 물체)'),
    ('Arise',   31.9,  '일어섬'),
    ('Walk',    walk_tmu_per_pace * walk_paces, f'10m 이동 ({walk_paces:.1f} paces, W-PO)'),
    ('M60B',    20.4 * weight_correction, '테이블에 내려놓기 위해 이동 (중량보정)'),
    ('RL1',      2.0,  '손 놓음'),
]

total_tmu = sum(tmu for _, tmu, _ in sequence)
total_sec = total_tmu * 0.036
people_sec = (total_tmu / 0.85) * 0.036

print(f"{'동작':<10} {'TMU':>8}  {'설명'}")
print("-" * 50)
for motion, tmu, desc in sequence:
    print(f"{motion:<10} {tmu:>8.1f}  {desc}")
print("-" * 50)
print(f"{'총 TMU':<10} {total_tmu:>8.1f}")
print(f"{'정미시간':<10} {total_sec:>8.2f} 초")
print(f"{'인간 기준':<10} {people_sec:>8.2f} 초 (×1/0.85)")

동작              TMU  설명
--------------------------------------------------
Bend           29.0  허리 숙임 (바닥 상자)
R30B           12.8  상자로 손 뻗음 (30cm 가정, Case B)
G4A             7.3  상자 잡음 (큰 물체)
Arise          31.9  일어섬
Walk          226.7  10m 이동 (13.3 paces, W-PO)
M60B           22.8  테이블에 내려놓기 위해 이동 (중량보정)
RL1             2.0  손 놓음
--------------------------------------------------
총 TMU         332.5
정미시간          11.97 초
인간 기준         14.08 초 (×1/0.85)


- 근데 왜 0.85를 곱하나요? 아 그건 간단합니다.
- 국제 노동기구에서 10%~15% 정도를 여유율로 잡고 있는데... 이게 뭔데요? 스트레칭, 장실타임, 담배타임(흡연자), 커피타임, 탕비실 털이 타임 다 감안해서 잡은겁니다.
- 그래서 저건 어떻게 해석하냐... 상기 동작을 수행하는데 이론상 11.97초가 걸리고 저기에 여유율을 적용하면 14.08초라는 얘기가 됩니다. 아, 그래서 한시간에 얼마나 하는지 궁금하시죠?

In [2]:
# 1시간이 100000TMU긴 한데 우리 초단위로 맞출거니까 초로 합시다.
robot = total_sec # 아틀라스
people = people_sec # 사람(여유율 적용됨)

robot_per_hour = 3600 / robot
people_per_hour = 3600 / people

print(f'Atlas Once: {robot:.2f}, {robot_per_hour:.2f} Times in 1hr | People once: {people:.2f}, {people_per_hour:.2f} Times in 1hr')

Atlas Once: 11.97, 300.74 Times in 1hr | People once: 14.08, 255.63 Times in 1hr


- 네. 저 상자 옮기는 일을 로봇은 1시간에 약 300번, 사람은 255번 한다는 얘깁니다. (소수점 이하 버림)

# 동작의 함수화
- 동작은 n킬로그램짜리 상자를 들어서 10미터 걸어가서 60cm 높이인 테이블에 내려놓는걸로 고정합니다. (안그러면 아주 복잡하게 바꿔야됨...)

## MTM calculate(MTM 도출하는 함수)

In [3]:
# 같은 동작을 상정하고 중량계수만 입력받습니다.
# 중량계수의 경우 가중치가 존재합니다.
def mtm_calculate(sc, value):
    # MTM-1 Data Card 기반 TMU 테이블
    reach_table = {
        30: {'A': 9.5, 'B': 12.8, 'C': 14.1, 'D': 11.7, 'E': 8.0},
        # 필요한 거리만 우선
    }

    move_table = {
        30: {'A': 12.7, 'B': 13.3, 'C': 15.1, 'Bm': 9.8},
        60: {'A': 22.1, 'B': 20.4, 'C': 25.2, 'Bm': 18.2},
    }

    grasp_table = {'G1A': 2.0, 'G1B': 3.5, 'G4A': 7.3}
    release_table = {'RL1': 2.0, 'RL2': 0.0}

    # 중량보정 (5kg)
    weight_correction = value  # Data Card 중량보정 계수

    # Walk: 15.0 TMU/pace, 1 pace ≈ 75cm → 10m = 약 13.3 paces
    walk_tmu_per_pace = 17.0  # 부하 있으므로 W-PO 사용
    walk_paces = 10 / 0.75

    # 작업 시퀀스
    sequence = [
        ('Bend',    29.0,  '허리 숙임'),
        ('R30B',    12.8,  '손 뻗음'),
        ('G4A',      7.3,  '잡기'),

        # 1. 일어설 때: (기본 31.9 * DF) + SC
        ('Arise',   (31.9 * weight_correction) + sc,  '중량물 들고 일어남 (허이짜!)'),

        ('Walk',    walk_tmu_per_pace * walk_paces, '10m 이동'),

        # 2. 내려놓을 때: (기본 20.4 * DF) + SC
        ('M60B',    (20.4 * weight_correction) + sc,  '테이블에 이동 (허이짜!)'),

        ('RL1',      2.0,  '손 놓음'),
    ]

    total_tmu = sum(tmu for _, tmu, _ in sequence)
    people_tmu = (total_tmu / 0.85)
    total_sec = total_tmu * 0.036
    people_sec = people_tmu * 0.036

    return total_tmu, people_tmu, total_sec, people_sec

In [4]:
total_tmu, people_tmu, total_sec, people_sec = mtm_calculate(4.3, 1.12)

print(f"{'동작':<10} {'TMU':>8}  {'설명'}")
print("-" * 50)
for motion, tmu, desc in sequence:
    print(f"{motion:<10} {tmu:>8.1f}  {desc}")
print("-" * 50)
print(f"{'총 TMU':<10} {total_tmu:>8.1f}")
print(f"{'총 TMU(사람)':<10} {people_tmu:>8.1f}")
print(f"{'정미시간':<10} {total_sec:>8.2f} 초")
print(f"{'인간 기준':<10} {people_sec:>8.2f} 초 (×1/0.85)")

동작              TMU  설명
--------------------------------------------------
Bend           29.0  허리 숙임 (바닥 상자)
R30B           12.8  상자로 손 뻗음 (30cm 가정, Case B)
G4A             7.3  상자 잡음 (큰 물체)
Arise          31.9  일어섬
Walk          226.7  10m 이동 (13.3 paces, W-PO)
M60B           22.8  테이블에 내려놓기 위해 이동 (중량보정)
RL1             2.0  손 놓음
--------------------------------------------------
총 TMU         344.9
총 TMU(사람)     405.8
정미시간          12.42 초
인간 기준         14.61 초 (×1/0.85)


## 나는 무게를 입력할터이니 너는 가중치를 찾거라
- 이거 사실 MTM 카드 봐야 하는겁니다. 근데 이걸 진짜로 카드 찾아서 하나하나 입력한다? 어느세월에 그걸 다 하고 있겠습니까...

In [5]:
# MTM card에 있는 가중치를 배열화
# 왼쪽: 무게/가운대: 정지상수(물건을 들 때 잠시 정지하는?)/오른쪽: 가중치
weight_data = [
    (1,  0.0,  1.00),
    (2,  1.6,  1.04),
    (4,  2.8,  1.07),
    (6,  4.3,  1.12),  # 5kg 상자가 속하는 구간
    (8,  5.8,  1.17),
    (10, 7.3,  1.22),
    (12, 8.8,  1.27),
    (14, 10.4, 1.32),
    (16, 11.9, 1.36),
    (18, 13.4, 1.41),
    (20, 14.9, 1.46),
    (22, 16.4, 1.51)
]

def get_weight_correction(weight):
    """입력된 무게에 맞는 SC와 DF를 반환합니다."""
    for limit, sc, df in weight_data:
        if weight <= limit:
            return sc, df
    return weight_data[-1][1], weight_data[-1][2] # 22kg 초과 시 최대치 적용

In [6]:
# 물건의 무게
goods_weight = [1, 5, 10, 15, 20, 25]
# 를 주면 가중치를 줘요 (주소 찾아가야되는건 단점)
for weight in goods_weight:
    if weight > 22:
        print(f"⚠️ 경고: {weight}kg은 MTM 표준 범위를 초과했습니다. (2인 1조 작업 권장)")
    # 그러면 우리는 뭘 하면 되냐... 그 가중치를
    sc, value = get_weight_correction(weight) # weight가 무게도 되고 가중치도 되면 난 변수명을 뭘로 지어야 하니
    # 여기에 넢으면 mtm을 줍니다.
    total_tmu, people_tmu, total_sec, people_sec = mtm_calculate(sc, value)

    # 출력하시면 됩니다.
    # 좀 간소화하고 싶으시면 몇줄은 주석처리해도 됩니다.
    print(f'무게: {weight}kg, 정지상수: {sc}, 가중치: {value}')
    print("-" * 50)
    print(f"{'총 TMU':<10} {total_tmu:>8.1f}")
    print(f"{'총 TMU(사람)':<10} {people_tmu:>8.1f}")
    print(f"{'정미시간':<10} {total_sec:>8.2f} 초")
    print(f"{'인간 기준':<10} {people_sec:>8.2f} 초 (×1/0.85)")
    print("=" * 50)

무게: 1kg, 정지상수: 0.0, 가중치: 1.0
--------------------------------------------------
총 TMU         330.1
총 TMU(사람)     388.3
정미시간          11.88 초
인간 기준         13.98 초 (×1/0.85)
무게: 5kg, 정지상수: 4.3, 가중치: 1.12
--------------------------------------------------
총 TMU         344.9
총 TMU(사람)     405.8
정미시간          12.42 초
인간 기준         14.61 초 (×1/0.85)
무게: 10kg, 정지상수: 7.3, 가중치: 1.22
--------------------------------------------------
총 TMU         356.2
총 TMU(사람)     419.0
정미시간          12.82 초
인간 기준         15.08 초 (×1/0.85)
무게: 15kg, 정지상수: 11.9, 가중치: 1.36
--------------------------------------------------
총 TMU         372.7
총 TMU(사람)     438.5
정미시간          13.42 초
인간 기준         15.78 초 (×1/0.85)
무게: 20kg, 정지상수: 14.9, 가중치: 1.46
--------------------------------------------------
총 TMU         383.9
총 TMU(사람)     451.7
정미시간          13.82 초
인간 기준         16.26 초 (×1/0.85)
⚠️ 경고: 25kg은 MTM 표준 범위를 초과했습니다. (2인 1조 작업 권장)
무게: 25kg, 정지상수: 16.4, 가중치: 1.51
-------------------------------------------

## 우리 그 시간도 바꿔보면 안됩니까?
- 단위는 초로 통일합니다.

In [7]:
# 1시간은 3600초입니다.
hour = 3600
# 30분, 1시간, 4시간, 8시간, 12시간, 22시간
hour_list = [0.5, 1, 4, 8, 12, 22]
# 초가 요기잉눼?
hour_sec_list = [int(hour * i) for i in hour_list]

In [8]:
# 이게 분모는 같고 분자가 달라지는거거든요? 그리고 그 분자가 MTM으로 계산한 시간인거예요.
# 일단 무게 고정합니닷 (정지상수+가중치 반영)
weight = 5

# 무게를 줄테니 알아서 계산하거라
sc, value = get_weight_correction(weight) # weight가 무게도 되고 가중치도 되면 난 변수명을 뭘로 지어야 하니
total_tmu, people_tmu, total_sec, people_sec = mtm_calculate(sc, value)

# 이렇게 하면 결과가 나오긴 나옵니다.
print(f'총 TMU {total_tmu:>8.1f} | 총 TMU(사람) {people_tmu:>8.1f}')
print(f'소요시간 {total_sec:>8.1f} | 소요시간(사람) {people_sec:>8.1f}')

# 위에서 계산한 시간을 소요시간으로 나눠주기만 하시면 되는 대단히 쉬운 작업입니다.
for i in hour_sec_list:
    robot_per_hour = i / total_sec # 로봇
    people_per_hour = i / people_sec # 사람

    # 출력
    print('=' * 50) # 구분선
    if i / 3600 < 1:
        print(f'{i // 60}분동안 진행할 수 있는 작업 횟수')
    else:
        print(f'{i // 3600}시간동안 진행할 수 있는 작업 횟수')# 시간을 쓰읍
    print(f'휴머노이드: {robot_per_hour:>8.1f}회 | 사람: {people_per_hour:>8.1f}회') # 단위시간 내에 할 수 있는 횟수(본론)

총 TMU    344.9 | 총 TMU(사람)    405.8
소요시간     12.4 | 소요시간(사람)     14.6
30분동안 진행할 수 있는 작업 횟수
휴머노이드:    145.0회 | 사람:    123.2회
1시간동안 진행할 수 있는 작업 횟수
휴머노이드:    289.9회 | 사람:    246.4회
4시간동안 진행할 수 있는 작업 횟수
휴머노이드:   1159.6회 | 사람:    985.7회
8시간동안 진행할 수 있는 작업 횟수
휴머노이드:   2319.2회 | 사람:   1971.3회
12시간동안 진행할 수 있는 작업 횟수
휴머노이드:   3478.8회 | 사람:   2957.0회
22시간동안 진행할 수 있는 작업 횟수
휴머노이드:   6377.9회 | 사람:   5421.2회


## 무게와 시간 둘 다 변수로 둬보자

In [9]:
# 1시간은 3600초입니다.
hour = 3600
# 30분, 1시간, 4시간, 8시간, 12시간, 22시간
hour_list = [0.5, 1, 4, 8, 12, 22]
# 초가 요기잉눼?
hour_sec_list = [int(hour * i) for i in hour_list] # 어차피 나눠봐야 정수인데 왜 플로트가 나오는겨

In [10]:
# 이게 분모는 같고 분자가 달라지는거거든요? 그리고 그 분자가 MTM으로 계산한 시간인거예요.
# 무게(들)
goods_weight = [1, 5, 10, 15, 20, 25] # MTM 리미트가 22킬로니까 여러분은 22킬로까지만 해보세요

for weight in goods_weight:
    # 무게를 줄테니 알아서 계산하거라
    sc, value = get_weight_correction(weight) # weight가 무게도 되고 가중치도 되면 난 변수명을 뭘로 지어야 하니
    total_tmu, people_tmu, total_sec, people_sec = mtm_calculate(sc, value)

    # 이렇게 하면 결과가 나오긴 나옵니다.
    print(f'총 TMU {total_tmu:>8.1f} | 총 TMU(사람) {people_tmu:>8.1f}')
    print(f'소요시간 {total_sec:>8.1f} | 소요시간(사람) {people_sec:>8.1f}')
    print("=" * 50)
    print(f'무게: {weight} | 정지상수: {sc} | 가중치: {value}')

    # 위에서 계산한 시간을 소요시간으로 나눠주기만 하시면 되는 대단히 쉬운 작업입니다.
    for i in hour_sec_list:
        robot_per_hour = i / total_sec # 로봇
        people_per_hour = i / people_sec # 사람

        # 출력
        print('-' * 50) # 구분선
        if i / 3600 < 1:
            print(f'{i // 60}분동안 진행할 수 있는 작업 횟수')
        else:
            print(f'{i // 3600}시간동안 진행할 수 있는 작업 횟수')# 시간을 쓰읍
        print(f'휴머노이드: {robot_per_hour:>8.1f}회 | 사람: {people_per_hour:>8.1f}회') # 단위시간 내에 할 수 있는 횟수(본론)
    print('\n')

총 TMU    330.1 | 총 TMU(사람)    388.3
소요시간     11.9 | 소요시간(사람)     14.0
무게: 1 | 정지상수: 0.0 | 가중치: 1.0
--------------------------------------------------
30분동안 진행할 수 있는 작업 횟수
휴머노이드:    151.5회 | 사람:    128.8회
--------------------------------------------------
1시간동안 진행할 수 있는 작업 횟수
휴머노이드:    303.0회 | 사람:    257.5회
--------------------------------------------------
4시간동안 진행할 수 있는 작업 횟수
휴머노이드:   1211.9회 | 사람:   1030.1회
--------------------------------------------------
8시간동안 진행할 수 있는 작업 횟수
휴머노이드:   2423.8회 | 사람:   2060.2회
--------------------------------------------------
12시간동안 진행할 수 있는 작업 횟수
휴머노이드:   3635.6회 | 사람:   3090.3회
--------------------------------------------------
22시간동안 진행할 수 있는 작업 횟수
휴머노이드:   6665.3회 | 사람:   5665.5회


총 TMU    344.9 | 총 TMU(사람)    405.8
소요시간     12.4 | 소요시간(사람)     14.6
무게: 5 | 정지상수: 4.3 | 가중치: 1.12
--------------------------------------------------
30분동안 진행할 수 있는 작업 횟수
휴머노이드:    145.0회 | 사람:    123.2회
--------------------------------------------------
1시간동안 진행할 수

# 여러분 우리 그 동작 하나만 더 합시다.

1. 높이가 90cm인 선반에서 길이가 50cm인 봉을 꺼낸 다음(무게는 변동)
2. 10미터 걸어가서요
3. 높이가 90cm인 랙에 수평으로(가로로) 거치할겁니다. 

In [11]:
# 길이는 가중치는 없는데요, 길이에 따라서 그게 있습니다. 그...
# 정밀도? 긴 봉같은거 들기 힘들잖아요.
def mtm_calculate_II(sc, df, length=50):
    # MTM-1 Data Card 기반 TMU 테이블
    reach_table = {
        30: {'A': 9.5, 'B': 12.8, 'C': 14.1, 'D': 11.7, 'E': 8.0},
        # 필요한 거리만 우선
    }

    move_table = {
        30: {'A': 12.7, 'B': 13.3, 'C': 15.1, 'Bm': 9.8},
        60: {'A': 22.1, 'B': 20.4, 'C': 25.2, 'Bm': 18.2},
    }

    grasp_table = {'G1A': 2.0, 'G1B': 3.5, 'G4A': 7.3}
    release_table = {'RL1': 2.0, 'RL2': 0.0}

    # 길이보정
    length_penalty = 0
    if length >= 50:
        # 긴 봉을 가로로 정렬해서 끼울 때 발생하는 '주춤거림' 보정 (약 9.1 TMU)
        length_penalty = 9.1
        move_case = 'C'
    else:
        move_case = 'B'

    # 중량보정 (5kg)
    weight_correction = df  # Data Card 중량보정 계수

    # Walk: 15.0 TMU/pace, 1 pace ≈ 75cm → 10m = 약 13.3 paces
    walk_tmu_per_pace = 17.0  # 부하 있으므로 W-PO 사용
    walk_paces = 10 / 0.75

    # 작업 시퀀스
    sequence = [
        # --- [1. 집기 섹션] ---
        ('R30B',    12.8,  '양손으로 부품 접근 (90cm 높이)'),
        ('G1B',      3.5,  '봉 형태 부품 파지 (양손 동시)'),
        ('M30B',    (13.3 * df) + sc, '선반에서 인출 (허이짜!)'),

        # --- [2. 이동 섹션] ---
        ('Walk',    17.0 * (10 / 0.75), '10m 이동 (부하 보행)'),

        # --- [3. 놓기 섹션] ---
        # 가로로 정렬해서 놓아야 하므로 일반 M보다 정밀한 Mc(Case C) 적용
        (f'M50{move_case}', (18.3 * df) + sc + length_penalty, f'랙 정렬 삽입 (길이 {length}cm 보정)'),
        ('RL1',      2.0,  '양손 놓기'),
    ]

    total_tmu = sum(tmu for _, tmu, _ in sequence)
    people_tmu = (total_tmu / 0.85)
    total_sec = total_tmu * 0.036
    people_sec = people_tmu * 0.036

    return total_tmu, people_tmu, total_sec, people_sec

In [12]:
mtm_calculate_II(4.3, 1.12)

(298.0586666666667, 350.6572549019608, 10.730112, 12.623661176470588)

## 바로 갑니다, 무게+시간으로!

In [13]:
# 1시간은 3600초입니다.
hour = 3600
# 30분, 1시간, 4시간, 8시간, 12시간, 22시간
hour_list = [0.5, 1, 4, 8, 12, 22]
# 초가 요기잉눼?
hour_sec_list = [int(hour * i) for i in hour_list] # 어차피 나눠봐야 정수인데 왜 플로트가 나오는겨

In [14]:
# 이게 분모는 같고 분자가 달라지는거거든요? 그리고 그 분자가 MTM으로 계산한 시간인거예요.
# 무게(들)
goods_weight = [1, 5, 10, 15, 20, 25] # MTM 리미트가 22킬로니까 여러분은 22킬로까지만 해보세요

for weight in goods_weight:
    # 무게를 줄테니 알아서 계산하거라
    sc, value = get_weight_correction(weight) # weight가 무게도 되고 가중치도 되면 난 변수명을 뭘로 지어야 하니
    total_tmu, people_tmu, total_sec, people_sec = mtm_calculate_II(sc, value)

    # 이렇게 하면 결과가 나오긴 나옵니다.
    print(f'무게: {weight} | 정지상수: {sc} | 가중치: {value}')
    print('-' * 50)
    print(f'총 TMU {total_tmu:>8.1f} | 총 TMU(사람) {people_tmu:>8.1f}')
    print(f'소요시간 {total_sec:>8.1f} | 소요시간(사람) {people_sec:>8.1f}')

    # 위에서 계산한 시간을 소요시간으로 나눠주기만 하시면 되는 대단히 쉬운 작업입니다.
    for i in hour_sec_list:
        robot_per_hour = i / total_sec # 로봇
        people_per_hour = i / people_sec # 사람

        # 출력
        print('-' * 50) # 구분선
        if i / 3600 < 1:
            print(f'{i // 60}분동안 진행할 수 있는 작업 횟수')
        else:
            print(f'{i // 3600}시간동안 진행할 수 있는 작업 횟수')# 시간을 쓰읍
        print(f'휴머노이드: {robot_per_hour:>8.1f}회 | 사람: {people_per_hour:>8.1f}회') # 단위시간 내에 할 수 있는 횟수(본론)
    print('\n')

무게: 1 | 정지상수: 0.0 | 가중치: 1.0
--------------------------------------------------
총 TMU    285.7 | 총 TMU(사람)    336.1
소요시간     10.3 | 소요시간(사람)     12.1
--------------------------------------------------
30분동안 진행할 수 있는 작업 횟수
휴머노이드:    175.0회 | 사람:    148.8회
--------------------------------------------------
1시간동안 진행할 수 있는 작업 횟수
휴머노이드:    350.1회 | 사람:    297.5회
--------------------------------------------------
4시간동안 진행할 수 있는 작업 횟수
휴머노이드:   1400.2회 | 사람:   1190.2회
--------------------------------------------------
8시간동안 진행할 수 있는 작업 횟수
휴머노이드:   2800.5회 | 사람:   2380.4회
--------------------------------------------------
12시간동안 진행할 수 있는 작업 횟수
휴머노이드:   4200.7회 | 사람:   3570.6회
--------------------------------------------------
22시간동안 진행할 수 있는 작업 횟수
휴머노이드:   7701.3회 | 사람:   6546.1회


무게: 5 | 정지상수: 4.3 | 가중치: 1.12
--------------------------------------------------
총 TMU    298.1 | 총 TMU(사람)    350.7
소요시간     10.7 | 소요시간(사람)     12.6
--------------------------------------------------
30분동안 진행할 수 있는